# CDR-MLC Oracle Expert Selection Diagnostics

This notebook measures whether CDR-MLC failures are caused by the router or by all experts failing.

## Strict interpretation

The oracle reads test labels **only after all expert predictions have been produced**. It is a diagnostic upper bound and must never be reported as deployable model performance. No oracle decision is used to train K-Means, the experts, or the evaluated CDR-MLC model.

The diagnostic also tests whether the oracle-selected expert is statistically predictable using only the same 15 congestion-window features. That cross-validation is a separability study performed on oracle labels—not a valid test result.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, cross_validate

try:
    display
except NameError:
    display = print
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

RANDOM_STATE = 42
BASE = Path('DATASETS/CDR-MLC/scale_1')
SCENARIOS = {
    'scenario_1': (BASE/'Short/level_1.csv', BASE/'Short/level_2.csv'),
    'scenario_2': (BASE/'Short/level_1.csv', BASE/'Short/level_3.csv'),
    'scenario_3': (BASE/'Short/level_2.csv', BASE/'Short/level_3.csv'),
    'scenario_4': (BASE/'Short/CDR-MLC-Shuffle.csv', BASE/'Long/CDR-MLC-Shuffle.csv'),
    'scenario_5': (BASE/'Long/CDR-MLC-Shuffle.csv', BASE/'Short/CDR-MLC-Shuffle.csv'),
}

main_notebook = json.loads(Path('CDR-MLC.ipynb').read_text(encoding='utf-8'))
exec(compile(''.join(main_notebook['cells'][0]['source']), 'CDR-MLC.ipynb::core', 'exec'), globals())
print('Loaded the exact leakage-safe CDR-MLC implementation')


In [ ]:
def oracle_expert_diagnostics(scenario, separability_sample=30000):
    train_path, test_path = SCENARIOS[scenario]
    print(f'\n{scenario}: {train_path} -> {test_path}')
    result = run_pipeline_from_two_files(
        train_file=str(train_path), test_file=str(test_path),
        n_clusters=3, window_size=3,
        clustering_stats=['mean','median','std','min','max']
    )
    test_df = result['test_df']
    features = result['classification_features']
    y_true = test_df[result['target_column']].to_numpy()
    current_routes = test_df['cluster'].to_numpy()

    # Every expert predicts every sample. Test labels are not read until afterward.
    expert_ids = np.array(sorted(result['classifiers']))
    all_predictions = np.column_stack([
        result['classifiers'][cid].predict(test_df[features]) for cid in expert_ids
    ])
    correct = all_predictions == y_true[:, None]
    correct_count = correct.sum(axis=1)
    any_correct = correct_count > 0
    current_column = np.searchsorted(expert_ids, current_routes)
    current_correct = correct[np.arange(len(test_df)), current_column]
    recoverable = (~current_correct) & any_correct
    irreducible = ~any_correct

    # Build a deterministic oracle target for analysis. Preserve the current route
    # when it is already correct; otherwise choose the nearest correct expert.
    routing_stats, _ = compute_sliding_window_stats(
        test_df, result['used_fixed_features'], result['window_size'], result['clustering_stats'])
    routing_vectors = result['scaler'].transform(routing_stats)
    distances = result['kmeans_model'].transform(routing_vectors)
    oracle_routes = current_routes.copy()
    for row in np.flatnonzero(recoverable):
        candidates = expert_ids[correct[row]]
        oracle_routes[row] = candidates[np.argmin(distances[row, candidates])]
    oracle_predictions = all_predictions[np.arange(len(test_df)), np.searchsorted(expert_ids, oracle_routes)]

    current_accuracy = float(accuracy_score(y_true, all_predictions[np.arange(len(test_df)), current_column]))
    oracle_accuracy = float(accuracy_score(y_true, oracle_predictions))
    summary = pd.DataFrame([{
        'scenario': scenario,
        'samples': len(test_df),
        'current_accuracy': current_accuracy,
        'oracle_any_expert_accuracy': oracle_accuracy,
        'maximum_recoverable_gain': oracle_accuracy-current_accuracy,
        'current_route_correct_%': 100*current_correct.mean(),
        'wrong_but_other_expert_correct_%': 100*recoverable.mean(),
        'no_expert_correct_%': 100*irreducible.mean(),
        'mean_correct_experts_per_sample': correct_count.mean(),
    }])
    print('\nOracle routing upper bound (diagnostic only)')
    display(summary.round(4))

    per_expert = []
    for col,cid in enumerate(expert_ids):
        per_expert.append({
            'expert': int(cid),
            'accuracy_if_used_for_all': accuracy_score(y_true, all_predictions[:,col]),
            'correct_samples': int(correct[:,col].sum()),
            'current_routed_samples': int((current_routes==cid).sum()),
            'oracle_target_samples': int((oracle_routes[any_correct]==cid).sum()),
        })
    per_expert = pd.DataFrame(per_expert)
    print('\nExpert capacity and routing demand')
    display(per_expert.round(4))

    routing_confusion = pd.DataFrame(
        confusion_matrix(oracle_routes[any_correct], current_routes[any_correct], labels=expert_ids),
        index=[f'oracle_{cid}' for cid in expert_ids],
        columns=[f'current_{cid}' for cid in expert_ids])
    print('\nOracle target versus current K-Means route')
    display(routing_confusion)

    # Diagnostic question: do the 15 routing features contain enough information
    # to distinguish oracle expert targets? This is NOT deployable performance.
    eligible = np.flatnonzero(any_correct)
    if len(eligible) > separability_sample:
        rng = np.random.default_rng(RANDOM_STATE)
        chosen = rng.choice(eligible, separability_sample, replace=False)
    else:
        chosen = eligible
    X_diag, y_diag = routing_vectors[chosen], oracle_routes[chosen]
    counts = pd.Series(y_diag).value_counts()
    separability = {'cv_accuracy_mean':np.nan, 'cv_balanced_accuracy_mean':np.nan,
                    'cv_f1_macro_mean':np.nan, 'samples':len(chosen)}
    if len(counts) >= 2 and counts.min() >= 5:
        folds = min(5, int(counts.min()))
        cv = StratifiedKFold(n_splits=folds, shuffle=True, random_state=RANDOM_STATE)
        diagnostic_router = RandomForestClassifier(
            n_estimators=80, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
        scores = cross_validate(
            diagnostic_router, X_diag, y_diag, cv=cv, n_jobs=1,
            scoring={'accuracy':'accuracy','balanced_accuracy':'balanced_accuracy','f1_macro':'f1_macro'})
        separability = {
            'cv_accuracy_mean':scores['test_accuracy'].mean(),
            'cv_balanced_accuracy_mean':scores['test_balanced_accuracy'].mean(),
            'cv_f1_macro_mean':scores['test_f1_macro'].mean(),
            'samples':len(chosen), 'folds':folds,
        }
    print('\n15-feature oracle-target separability (diagnostic CV; not model performance)')
    display(pd.DataFrame([separability]).round(4))

    count_table = pd.Series(correct_count).value_counts().sort_index().rename_axis('number_of_correct_experts').reset_index(name='samples')
    count_table['share_%'] = 100*count_table['samples']/len(test_df)
    print('\nHow many experts are correct for each sample?')
    display(count_table.round(4))
    count_table.set_index('number_of_correct_experts')['share_%'].plot.bar(
        figsize=(7,4), rot=0, title=f'{scenario}: correct expert availability')
    plt.ylabel('Test samples (%)'); plt.grid(axis='y',alpha=.25); plt.tight_layout(); plt.show()

    return {'base_result':result, 'summary':summary, 'per_expert':per_expert,
            'routing_confusion':routing_confusion, 'separability':separability,
            'correct_matrix':correct, 'oracle_routes':oracle_routes,
            'current_routes':current_routes, 'recoverable_mask':recoverable,
            'irreducible_mask':irreducible, 'routing_vectors':routing_vectors}


In [ ]:
# Scenario 1: run independently
scenario_1_oracle = oracle_expert_diagnostics('scenario_1')


In [ ]:
# Scenario 2: run independently
scenario_2_oracle = oracle_expert_diagnostics('scenario_2')


In [ ]:
# Scenario 3: run independently
scenario_3_oracle = oracle_expert_diagnostics('scenario_3')


In [ ]:
# Scenario 4: run independently
scenario_4_oracle = oracle_expert_diagnostics('scenario_4')


In [ ]:
# Scenario 5: run independently
scenario_5_oracle = oracle_expert_diagnostics('scenario_5')
